# Practice Lab: When to Fine-Tune (Lesson 9.1)

Module 9 · 8 exercises · Decision tree + cost calculator + data curation + regression testing

Exercises 1, 2, 4, 5, 6, 7 run locally (pure Python).


# Lesson 9.1: When to Fine-Tune — Decision Framework

8 exercises: 2E + 3M + 3C

Exercises 1, 2, 4, 5, 6, 7 run **locally** (pure Python). Ex 3, 8 are design/analysis.


In [ ]:
import numpy as np
np.random.seed(42)


---
## Exercise 1 (Easy): Decision Tree Classifier


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
def should_ft(know, changes, style, tried, data, vol=0):
    if know and changes: return "RAG","Dynamic data -> RAG"
    if not tried: return "PROMPT","Try prompting first"
    if know and not changes and vol > 10000: return "FINE-TUNE","Static+volume -> bake in"
    if know and not changes: return "RAG","Static+low vol -> RAG"
    if style and data: return "FINE-TUNE","Behavior+data -> FT"
    if style and not data: return "PROMPT","Style but no data -> few-shot"
    return "PROMPT","Default: prompting"

scenarios = [
    ("Support bot (FAQ changes)",True,True,False,True,False,"RAG"),
    ("Code reviewer style",False,False,True,True,True,"FINE-TUNE"),
    ("Legal summarizer (1st try)",False,False,True,False,False,"PROMPT"),
    ("Medical classifier 100K/day",True,False,True,True,True,"FINE-TUNE"),
    ("Company policy chatbot",True,True,False,True,False,"RAG"),
    ("JSON formatter",False,False,False,False,False,"PROMPT"),
    ("Hindi clinical notes",False,False,True,True,True,"FINE-TUNE"),
    ("News summarizer (real-time)",True,True,False,True,False,"RAG"),
    ("Email tone adjuster",False,False,True,False,True,"PROMPT"),
    ("Product catalog",True,False,False,True,False,"RAG"),
]

print("Decision Tree (10 Scenarios):")
ok = 0; counts = {}
for name,k,c,s,t,d,exp in scenarios:
    r,reason = should_ft(k,c,s,t,d,100000 if "100K" in name else 500)
    match = r == exp
    if match: ok += 1
    counts[r] = counts.get(r,0) + 1
    print(f"  [{'OK' if match else '!!'}] {name} -> {r}: {reason}")

print(f"\nAccuracy: {ok}/{len(scenarios)}")
print(f"Distribution: {counts}")
print(f"Theory: PROMPT 70% | RAG 25% | FT 5%")


</details>


---
## Exercise 2 (Easy): Cost-Benefit Calculator


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
class CostCalc:
    def __init__(self, cpd, months=6, tin=500, tout=200):
        self.cpd=cpd; self.days=months*30; self.tin=tin; self.tout=tout
        self.pi=0.10; self.po=0.40  # Gemini Flash $/MTok
    def _api(self, extra=0):
        pc = ((self.tin+extra)*self.pi + self.tout*self.po)/1e6
        return pc*self.cpd*self.days, pc
    def prompt(self):
        t,pc = self._api(200); return {"m":"Prompt","setup":0,"total":t,"pc":pc}
    def rag(self):
        t,pc = self._api(800); t += 50*(self.days/30)+200; return {"m":"RAG","setup":200,"total":t,"pc":pc}
    def ft(self):
        t,pc = self._api(0); t += 550+50*(self.days/30/3); return {"m":"FT","setup":550,"total":t,"pc":pc}
    def compare(self):
        ms = sorted([self.prompt(),self.rag(),self.ft()], key=lambda x:x["total"])
        print(f"\n  {self.cpd:,}/day, {self.days//30} months:")
        for m in ms:
            w = " *" if m==ms[0] else ""
            print(f"    {m['m']:<8} setup=${m['setup']:<6} total=${m['total']:>9.2f} per_call=${m['pc']:.6f}{w}")
        return ms[0]["m"]

print("Cost Calculator:")
for vol in [100, 5000, 100000]:
    w = CostCalc(vol).compare()

print(f"\nBreak-even: FT wins at high volume (setup recovered by per-call savings)")


</details>


---
## Exercise 4 (Medium): Data Curation Pipeline


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import numpy as np
np.random.seed(42)

def score_example(inp, out):
    iw = set(inp.lower().split()) - {"what","is","the","a","how","for","in","to"}
    ow = set(out.lower().split())
    rel = min(len(iw&ow)/max(len(iw),1), 1.0)
    cor = min(len(out.split())/8, 1.0)
    comp = min(len(out)/max(len(inp),1)*0.5, 1.0)
    fmt = 0.6 + (0.2 if "." in out else 0) + (0.2 if out and out[0].isupper() else 0)
    return round((rel+cor+comp+fmt)/4, 2)

examples = [
    ("Refund policy?","Full refund within 7 days. 50% from 8-30 days. None after 30."),
    ("Course cost?","14999"),
    ("Prerequisites?","Basic Python and high school math. No prior ML needed."),
    ("Deployment?","ok"),
    ("Tools learned?","Google Colab, Vertex AI, ChromaDB, LangChain, Docker."),
    ("EMI available?","Yes, EMI via Razorpay starting at 2500 rupees per month."),
    ("Live classes?","Daily at 7 PM IST. Recordings available within 2 hours."),
    ("Certificate?","yes"),
    ("Duration?","146 hours across 14 modules covering Python, ML, GenAI, deployment."),
    ("Group discount?","20% off for groups of 3 or more students."),
]

scored = [(i,o,score_example(i,o)) for i,o in examples]
scored.sort(key=lambda x:-x[2])
cutoff = int(len(scored)*0.8)

print("Data Curation:")
print(f"\nKEPT ({cutoff}):")
for i,o,s in scored[:cutoff]: print(f"  [{s:.2f}] {i[:25]}... -> {o[:30]}...")
print(f"\nDROPPED ({len(scored)-cutoff}):")
for i,o,s in scored[cutoff:]: print(f"  [{s:.2f}] {i[:25]}... -> {o[:30]}...")

all_s = [s for _,_,s in scored]
print(f"\nMean quality: {np.mean(all_s):.2f} -> Kept mean: {np.mean([s for _,_,s in scored[:cutoff]]):.2f}")
print(f"LIMA: 1000 curated > 52K raw. Quality >> quantity.")


</details>


---
## Exercise 5 (Medium): Method Selection


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
def select(task, data, goal):
    if "reason" in task or "math" in task: return "GRPO","Verifiable rewards"
    if "distill" in goal or "cost" in goal: return "Distillation","Large->small"
    if "domain" in goal: return "Continual PT","New domain knowledge"
    if "preference" in data: return "DPO","Preference pairs"
    if "binary" in data or "thumb" in data: return "KTO","Binary feedback"
    if "style" in goal: return "SimPO","Style without ref model"
    return "SFT","Input-output pairs"

tests = [
    ("Clinical formatter","classification","io pairs","format","SFT"),
    ("Support tone","generation","preference pairs","tone","DPO"),
    ("Math solver","reasoning","verifiable","accuracy","GRPO"),
    ("Replace GPT-4o","generation","teacher out","cost distill","Distillation"),
    ("Hindi legal NER","classification","io pairs","domain knowledge","Continual PT"),
    ("Chatbot feedback","generation","binary thumbs","helpfulness","KTO"),
    ("Code review","generation","preference pairs","style","DPO"),
    ("Invoice parser","extraction","io pairs","accuracy","SFT"),
]

print("Method Selection:")
ok = 0
for name,task,data,goal,exp in tests:
    m,r = select(task,data,goal)
    match = m == exp
    if match: ok += 1
    print(f"  [{'OK' if match else '!!'}] {name} -> {m}: {r}")

print(f"\nAccuracy: {ok}/{len(tests)}")
print(f"\nRule: SFT=format, DPO=preferences, GRPO=reasoning, Distill=cost")



</details>


---
## Exercise 6 (Challenge): Synthetic Data Pipeline


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import numpy as np
np.random.seed(42)

def gen_synth(topic, n=5):
    tmpls = [
        (f"What is {topic}?", f"{topic} is a technique for improving AI model performance in production."),
        (f"How does {topic} work?", f"{topic} works by processing data through specialized transformations for better results."),
        (f"When to use {topic}?", f"Use {topic} when you have sufficient training data and need behavioral changes."),
        (f"Compare {topic} with alternatives.", f"{topic} vs alternatives: offers higher accuracy but requires more compute."),
        (f"Limitations of {topic}?", f"Key limitations: requires quality data and risk of overfitting. Mitigate with curation."),
    ]
    return [{"i":q,"o":a} for q,a in tmpls[:n]]

def score_q(ex):
    out = ex["o"]; words = len(out.split())
    ls = min(words/20,1.0) if words<60 else 0.8
    ds = len(set(out.lower().split()))/max(words,1)
    inp_w = set(ex["i"].lower().split()) - {"what","is","how","when","to","use"}
    rs = len(inp_w & set(out.lower().split()))/max(len(inp_w),1)
    return round((ls+ds+rs)/3, 3)

all_ex = []
for t in ["fine-tuning","LoRA","distillation","DPO"]:
    all_ex.extend(gen_synth(t))

scored = sorted([(e,score_q(e)) for e in all_ex], key=lambda x:-x[1])
cut = int(len(scored)*0.6)
kept, dropped = scored[:cut], scored[cut:]

print("Synthetic Data Pipeline:")
print(f"Generated: {len(all_ex)} from 4 topics")
print(f"\nTop 3 kept:")
for e,s in kept[:3]: print(f"  [{s:.3f}] {e['i'][:35]}...")
print(f"\nBottom 2 dropped:")
for e,s in dropped[-2:]: print(f"  [{s:.3f}] {e['i'][:35]}...")

all_s = [s for _,s in scored]; kept_s = [s for _,s in kept]
print(f"\nAll mean={np.mean(all_s):.3f} | Kept mean={np.mean(kept_s):.3f}")
print(f"Improvement: {(np.mean(kept_s)/np.mean(all_s)-1)*100:.1f}%")
print(f"AlpaGasus: 9K filtered > 52K raw. Always filter!")


</details>


---
## Exercise 7 (Challenge): Regression Test Suite


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
class RegSuite:
    CATS = {"target":(0.30,0.85),"knowledge":(0.20,0.70),"reasoning":(0.20,0.65),"safety":(0.15,0.90),"format":(0.15,0.85)}
    def eval(self, before, after):
        for cat,(w,th) in self.CATS.items():
            b,a = before[cat],after[cat]; d = a-b
            st = "PASS" if a>=th else "FAIL"
            fg = " !! FORGETTING" if d<-0.05 else ""
            print(f"  [{st}] {cat:<12} {b:.2f}->{a:.2f} ({d:+.2f}){fg}")

suite = RegSuite()
before = {"target":0.72,"knowledge":0.82,"reasoning":0.78,"safety":0.95,"format":0.88}
after = {"target":0.94,"knowledge":0.68,"reasoning":0.55,"safety":0.92,"format":0.90}

print("Regression Test Suite:")
suite.eval(before, after)
forgot = {k:after[k]-before[k] for k in before if after[k]-before[k]<-0.05}
print(f"\nForgetting in: {list(forgot.keys())}")
print(f"Mitigations: data mixing, low LR, LoRA, model merging, early stopping")


</details>


---
## Exercise 3 (Medium): Prompt Baseline Methodology
Design/analysis. See practice-lab-9_1.html.


In [ ]:
# Exercise 3: Prompt Baseline Methodology
pass


---
## Exercise 8 (Challenge): Full Decision Audit
Design/analysis. See practice-lab-9_1.html.


In [ ]:
# Exercise 8: Full Decision Audit
pass
